# FLEXPART historical black carbon over Europe

This notebook explores the modelled presence of black carbon (BC) over Europe for a location and a
year of your choice, separating the **wildfire** contribution from the **anthropogenic** one.

Daily GFAS/CAMS global biomass-burning emission fluxes are used to model wildfire aerosol. FLEXPART
analyses are available as daily fields at 14 height levels, on a 0.5° × 0.5° grid from 2016 and
1° × 1° before, covering **2008–2025**. Data are read directly from the NILU THREDDS server, so
nothing has to be downloaded by hand:
[thredds.nilu.no/thredds/catalog/flexpart/IRISCC](https://thredds.nilu.no/thredds/catalog/flexpart/IRISCC/catalog.html)

Two figures are produced for the selected point:

* the **surface concentration** over the year, as a stacked plot of the anthropogenic and wildfire
  contributions;
* a **vertical cross-section** of the total BC concentration, time against height.

*Model and data by Sabine Eckhardt and Nikolaos Evangeliou (NILU).*

**Folder management: to avoid accumulating files across repeated runs, this notebook empties its `outputs/` and `exports/` folders at the start of each run. Use the export cell at the end to keep anything you want before running it again.**

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

## 1. Dependencies

Only the packages that are actually missing are installed, so this costs nothing on an environment
that already has them. Each VRE user has their own container, so a notebook that runs for a
colleague can still fail here with `ModuleNotFoundError`.

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

import os
import importlib.util
import subprocess
import sys

REQUIRED = {
    "threddsclient": "threddsclient",
    "netCDF4": "netCDF4",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "ipywidgets": "ipywidgets",
    "requests": "requests",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if not missing:
    print("--> All required packages are already installed.")
else:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    if result.returncode == 0:
        print("--> Done. If an import still fails below, restart the kernel and run again.")
    else:
        print("--> Installation failed:")
        print(result.stderr[-1500:])
        if "externally-managed-environment" in result.stderr:
            print(f"\n--> This environment does not allow pip to install into it directly.")
            print(f"    Ask the VRE administrators, or retry with:")
            print(f"    {sys.executable} -m pip install --user " + " ".join(missing))

In [ ]:
# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "FLEXPART historical black carbon"
NOTEBOOK_VERSION = "2.1"
NOTEBOOK_DATE = "2026-10-25"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

from datetime import datetime, timedelta
from pathlib import Path
import time

import numpy as np
import requests
import threddsclient
from netCDF4 import Dataset

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm, Normalize

import ipywidgets as widgets
from IPython.display import display

# Bump this whenever the notebook is updated, so you can tell at a glance which version the
# kernel is actually running (a stale kernel keeps old definitions after the file is updated).

# Note: the original script also imported mpl_toolkits.basemap. Nothing in the plotting code
# uses it, and Basemap is deprecated and awkward to install, so it is left out here.

NOTEBOOK_SLUG = "flexpart"

# --- Outputs of this run -------------------------------------------------------------
# Figures are shown inline and kept in memory; nothing is written to disk while you work.
# The export cell at the end of the notebook writes out what you decide to keep, into the
# folder below, and packs it into a single zip you can download.
#
# The folder is emptied at the start of each run, so that repeated runs do not accumulate
# files. Set CLEAN_OUTPUTS to False to keep what earlier runs produced.
import shutil

OUTPUT_DIR = Path("outputs")
CLEAN_OUTPUTS = True

if CLEAN_OUTPUTS and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRAWN_FIGURES = {}

def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Choose the location and the year

Pick a station from the list to fill in its coordinates, or choose **Custom coordinates** and type
them yourself. The FLEXPART grid is 0.5° (1° before 2016), so the nearest grid point is used — the
exact decimals matter little.

Press **Confirm selection** when you are done.

In [ ]:
DEFAULT_NAME, DEFAULT_LON, DEFAULT_LAT, DEFAULT_YEAR = "Zeppelin", 11.89, 78.91, 2022
YEARS = list(range(2008, 2026))

# Station list from the ACTRIS-ARES API, only to prefill coordinates. It is optional: if the
# service is unreachable the notebook still works with coordinates typed by hand.
station_options = [("Custom coordinates", None)]
try:
    _r = requests.get("https://api.actris-ares.eu/api/services/restapi/stations", timeout=30)
    _r.raise_for_status()
    _stations = {s["ID"]: s for s in _r.json()}
    station_options += sorted(
        (f"{s['ID']} - {s.get('Location', '')}", s["ID"]) for s in _stations.values()
    )
    print(f"{len(_stations)} ACTRIS stations available to prefill the coordinates.")
except Exception as exc:  # noqa: BLE001 - the station list is a convenience, not a requirement
    _stations = {}
    print(f"Station list unavailable ({type(exc).__name__}); enter the coordinates by hand.")

station_dd = widgets.Dropdown(options=station_options, value=None, description="Station:",
                              style={"description_width": "initial"},
                              layout=widgets.Layout(width="520px"))
name_tx = widgets.Text(value=DEFAULT_NAME, description="Name:",
                       style={"description_width": "initial"},
                       layout=widgets.Layout(width="360px"))
lon_tx = widgets.FloatText(value=DEFAULT_LON, description="Longitude:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="260px"))
lat_tx = widgets.FloatText(value=DEFAULT_LAT, description="Latitude:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="260px"))
year_dd = widgets.Dropdown(options=YEARS, value=DEFAULT_YEAR, description="Year:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="200px"))
confirm_btn = widgets.Button(description="Confirm selection", button_style="success", icon="check")
confirm_out = widgets.Output()

if "SELECTION" not in globals():
    SELECTION = None


def _on_station(change):
    """Fill the coordinate boxes from the station list; they stay editable."""
    s = _stations.get(change["new"])
    if s:
        name_tx.value = s.get("Location") or s["ID"]
        lon_tx.value = float(s["Longitude"])
        lat_tx.value = float(s["Latitude"])


def _on_confirm(_):
    global SELECTION
    SELECTION = dict(name=name_tx.value.strip() or "point",
                     lon=float(lon_tx.value), lat=float(lat_tx.value), year=int(year_dd.value))
    with confirm_out:
        confirm_out.clear_output()
        print(f"Confirmed: {SELECTION['name']} "
              f"({SELECTION['lat']:.2f} N, {SELECTION['lon']:.2f} E), {SELECTION['year']}")


station_dd.observe(_on_station, names="value")
confirm_btn.on_click(_on_confirm)
display(widgets.VBox([station_dd, name_tx,
                      widgets.HBox([lon_tx, lat_tx]), year_dd, confirm_btn, confirm_out]))

with confirm_out:
    if SELECTION:
        print(f"Confirmed: {SELECTION['name']} ({SELECTION['lat']:.2f} N, "
              f"{SELECTION['lon']:.2f} E), {SELECTION['year']} (already confirmed)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_NAME}, {DEFAULT_YEAR}.")

## 3. Cross-section scale

The colour scale of the vertical cross-section can be linear or logarithmic. A logarithmic scale
helps when a few strong plumes would otherwise flatten everything else; `min_v` must then be
strictly positive.

Keeping the figure and the time series is decided at the end, in the
[export cell](#export) — there is nothing to set here for that.

In [ ]:
log_dd = widgets.Dropdown(options=[("Linear", False), ("Logarithmic", True)], value=False,
                          description="Colour scale:", style={"description_width": "initial"},
                          layout=widgets.Layout(width="300px"))
logbase_dd = widgets.Dropdown(options=[2, 10], value=2, description="Log base:",
                              style={"description_width": "initial"},
                              layout=widgets.Layout(width="200px"))
minv_tx = widgets.FloatText(value=1.0, description="min value:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="220px"))
maxv_tx = widgets.FloatText(value=256.0, description="max value:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="220px"))
freq_tx = widgets.FloatText(value=20.0, description="tick step (linear only):",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="320px"))
display(widgets.VBox([log_dd, logbase_dd, widgets.HBox([minv_tx, maxv_tx]), freq_tx]))
print("These are read when the figure is drawn, further down; no need to confirm them here.")

## 4. Locate the file on the THREDDS server

One NetCDF file per year holds both contributions, as the variables `BC_conc_BB` (biomass burning)
and `BC_conc_ANT` (anthropogenic).

In [ ]:
SELECTION = globals().get("SELECTION")
if not SELECTION:
    SELECTION = dict(name=DEFAULT_NAME, lon=DEFAULT_LON, lat=DEFAULT_LAT, year=DEFAULT_YEAR)
    print(f"No selection confirmed - defaulting to {DEFAULT_NAME}, {DEFAULT_YEAR}.")

station_name, target_lon, target_lat, year = (SELECTION["name"], SELECTION["lon"],
                                              SELECTION["lat"], SELECTION["year"])
print(f"Point: {station_name} ({target_lat:.2f} N, {target_lon:.2f} E), year {year}")

CATALOG = "https://thredds.nilu.no/thredds/catalog/flexpart/IRISCC/catalog.html"
t0 = time.time()
try:
    histFP_urls = threddsclient.opendap_urls(CATALOG)
except Exception as exc:  # noqa: BLE001 - a network failure should be readable
    raise RuntimeError(
        f"Could not read the THREDDS catalog ({type(exc).__name__}: {exc}). "
        f"Check your connection and that {CATALOG} is reachable."
    ) from exc
print(f"{len(histFP_urls)} dataset(s) in the catalog ({time.time() - t0:.1f} s).")

filebc = next((f for f in histFP_urls
               if f.endswith(f"_{year}.nc") and "FLEXPART_BC_" in f), None)

if filebc is None:
    # Say which years are actually there rather than failing with StopIteration
    years = sorted({f[-7:-3] for f in histFP_urls if "FLEXPART_BC_" in f and f.endswith(".nc")})
    raise RuntimeError(f"No FLEXPART_BC file for {year}. Years available: {', '.join(years)}")

print("Reading file:", filebc.rsplit("/", 1)[-1])

## 5. Read the profile at the selected point

Only the column above the nearest grid point is read, in chunks along time, so the whole field never
has to be transferred. This is the slowest step of the notebook: a full year is a few thousand time
steps.

In [ ]:
def read_point_profile(fname, target_lon, target_lat, chunk_t=24):
    """Read the BC time-height profile above the grid point nearest to the target.

    Returns (lons, lats, times, heights, ant, bb, idZy, idZx) with ant and bb of shape (nt, nz).
    """
    with Dataset(fname, "r") as nc:
        for v in ("longitude", "latitude", "time", "altitude", "BC_conc_BB", "BC_conc_ANT"):
            try:
                nc[v].set_auto_mask(False)
            except Exception:
                pass

        lons, lats = nc["longitude"][:], nc["latitude"][:]
        times_raw, hei = nc["time"][:], nc["altitude"][:]

        idZx = int(np.abs(lons - target_lon).argmin())
        idZy = int(np.abs(lats - target_lat).argmin())
        # Kept and shown on the figures: two nearby requests can land on different grid cells,
        # which is the usual reason why two runs of this notebook disagree.
        global grid_lon, grid_lat
        grid_lon, grid_lat = float(lons[idZx]), float(lats[idZy])
        print(f"Nearest grid point: lon {grid_lon:.2f}, lat {grid_lat:.2f} "
              f"(indices {idZy}, {idZx}); requested lon {target_lon:.2f}, lat {target_lat:.2f}")

        mr_BB, mr_ANT = nc["BC_conc_BB"], nc["BC_conc_ANT"]
        nt, nz, ny, nx = mr_BB.shape
        if not (0 <= idZy < ny and 0 <= idZx < nx):
            raise IndexError(f"Point ({idZy},{idZx}) outside the grid ({ny},{nx}). "
                             f"The requested coordinates fall outside the model domain.")

        bb = np.empty((nt, nz), dtype=np.float32)
        ant = np.empty((nt, nz), dtype=np.float32)
        for t0 in range(0, nt, chunk_t):
            t1 = min(t0 + chunk_t, nt)
            bb[t0:t1] = mr_BB[t0:t1, :, idZy, idZx].astype(np.float32, copy=False)
            ant[t0:t1] = mr_ANT[t0:t1, :, idZy, idZx].astype(np.float32, copy=False)
            if t0 % (chunk_t * 40) == 0:
                print(f"  {t1}/{nt} time steps", end="\r", flush=True)

    base = datetime(1970, 1, 1)
    times = [base + timedelta(days=float(t)) for t in times_raw]
    return lons, lats, times, hei, ant, bb, idZy, idZx


t0 = time.time()
lons, lats, tt, hei, nct_ant, nct_bb, idZy, idZx = read_point_profile(
    filebc, target_lon, target_lat, chunk_t=24)
print(f"\nRead {nct_ant.shape[0]} time steps x {nct_ant.shape[1]} levels "
      f"in {time.time() - t0:.0f} s.")

## 6. Plot

The upper panel stacks the anthropogenic and wildfire contributions at the surface; the lower one
shows the vertical cross-section of their sum.

In [ ]:
def lineplot_full(ax, dat, ant, bb, level=0, label1="Anthropogenic", label2="Wildfire",
                  ylabel=r"Surface BC (ng m$^{-3}$)", title="", ylim=None):
    """Stacked area plot of the two contributions at one level."""
    a = np.asarray(ant)[:, level]
    b = np.asarray(bb)[:, level]
    if not (len(dat) == len(a) == len(b)):
        raise ValueError("Time and data lengths do not match")

    ax.fill_between(dat, 0.0, a, alpha=0.7, label=label1)
    ax.fill_between(dat, a, a + b, alpha=0.7, label=label2)
    total = a + b
    ax.plot(dat, total, color="k", linewidth=1.2)

    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=14)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=15))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    ax.tick_params(axis="x", rotation=30)
    ax.set_xlim(dat[0], dat[-1])

    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        ymax = np.nanmax(total)
        ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1.0)
    ax.legend(frameon=False)


def cross(fig, ax, tt, hei, nct, ylabel=r"Height (km)", title="",
          log=False, log_base=2, min_v=0.0, max_v=6.0, freq=0.5):
    """Time-height cross-section of the concentration."""
    cmap = plt.get_cmap("rainbow")
    x, y = np.meshgrid(tt, np.asarray(hei) * 1e-3)

    Z = np.asarray(nct)
    if Z.shape != x.shape:
        Z = Z.T
    if Z.shape != x.shape:
        raise ValueError(f"Data shape {Z.shape} does not match the grid {x.shape}.")

    ax.clear()
    if log:
        min_v, max_v = float(min_v), float(max_v)
        if not (min_v > 0 and max_v > min_v):
            raise ValueError("A logarithmic scale needs 0 < min value < max value.")
        Z = np.asarray(Z, dtype=float)
        Z[Z <= 0] = min_v                      # keep log plotting from failing on zeros
        norm = LogNorm(vmin=min_v, vmax=max_v)
        pmin, pmax = np.log(min_v) / np.log(log_base), np.log(max_v) / np.log(log_base)
        clevs = np.power(float(log_base), np.linspace(pmin, pmax, 256))
        ticks = np.power(float(log_base),
                         np.arange(int(np.floor(pmin)), int(np.ceil(pmax)) + 1, dtype=float))
        ticks = ticks[(ticks >= min_v) & (ticks <= max_v)]
    else:
        min_v, max_v = 0.0, float(max_v)
        norm = Normalize(vmin=min_v, vmax=max_v)
        clevs = np.linspace(min_v, max_v, 256)
        ticks = np.arange(min_v, max_v + freq, freq, dtype=float)

    cs1 = ax.contourf(x, y, Z, clevs, cmap=cmap, extend="both", norm=norm)

    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(tt[0], tt[-1])
    ax.set_ylim(0, 20.01)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=15))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    ax.tick_params(axis="x", rotation=30)

    # Remove colorbars left by a previous run of this cell, otherwise they stack up
    for cax in getattr(ax, "_cross_cbar_axes", []):
        try:
            cax.remove()
        except Exception:
            pass
    ax._cross_cbar_axes = []

    bbox = ax.get_position()
    cb_height, cb_pad = 0.05 * bbox.height, 0.15 * bbox.height
    cbaxes = fig.add_axes([bbox.x0, bbox.y0 - cb_pad - cb_height, bbox.width, cb_height])
    ax._cross_cbar_axes.append(cbaxes)

    cbar = fig.colorbar(cs1, cax=cbaxes, ticks=ticks, orientation="horizontal")
    cbar.minorticks_off()
    cbar.set_ticklabels([f"{t:g}" for t in ticks])
    cbar.set_label(r"Vertical BC (ng m$^{-3}$)", fontsize=14)
    return cs1


# The grid point is part of the titles: the values depend on which 0.5 deg cell was used, so
# a figure that does not say which one cannot be compared with another run.
where = f"{station_name} ({grid_lat:.2f}N, {grid_lon:.2f}E) in {year}"

fig = plt.figure(figsize=(10, 10))
ax1 = fig.add_subplot(211)
lineplot_full(ax1, tt, nct_ant, nct_bb, level=0,
              title=f"Surface concentration of Black Carbon at {where}")

ax2 = fig.add_subplot(212)
cross(fig, ax2, tt, hei, nct_ant + nct_bb,
      title=f"Vertical cross-section of Black Carbon at {where}",
      log=log_dd.value, log_base=logbase_dd.value,
      min_v=minv_tx.value, max_v=maxv_tx.value, freq=freq_tx.value)

register_figure(fig, f"{station_name}_{year}")
plt.show()

<a id="export"></a>
## Take the results with you

The figures above are only displayed in the notebook: nothing has been written to disk while you
worked, and the output folder is emptied every time the notebook is run again.

The cell below writes out the figure and the surface time series, packs them into a single zip and
gives you a download link. Run it before starting a new run if you want to keep anything.

The archive is written into an `exports/` folder next to the notebook. That folder is emptied at the start of each run as well, so download what you need before running the notebook again.

In [ ]:
import os
import zipfile

from IPython.display import HTML

RESULT_SUFFIXES = (".png", ".csv", ".txt")


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


bundle_btn = widgets.Button(description="Create archive", button_style="success",
                            icon="file-archive-o")
bundle_out = widgets.Output()


def _make_bundle(_=None):
    with bundle_out:
        bundle_out.clear_output()

        # The figures live in memory only: the inline backend closes them at the end of each
        # cell, so they are written out here rather than while plotting.
        written = []
        for stem, fig in DRAWN_FIGURES.items():
            path = OUTPUT_DIR / f"{stem}.png"
            if not path.exists():
                fig.savefig(path, dpi=150, bbox_inches="tight")
                written.append(path.name)
        if written:
            print(f"Wrote {len(written)} figure(s): {', '.join(written)}")

        # The surface time series is the other thing worth keeping from a run
        txt_path = OUTPUT_DIR / f"{station_name.replace(' ', '_')}_{year}.txt"
        if not txt_path.exists():
            with open(txt_path, "w") as fh:
                fh.write("date time anthropogenic wildfire total\n")
                for t, v1, v2 in zip(tt, nct_ant[:, 0], nct_bb[:, 0]):
                    fh.write(f"{t:%Y-%m-%d %H:%M:%S} {v1:.6e} {v2:.6e} {v1 + v2:.6e}\n")
            print(f"Wrote the surface time series: {txt_path.name} ({len(tt)} rows)")

        files = [f for f in sorted(OUTPUT_DIR.rglob("*"))
                 if f.is_file() and f.suffix.lower() in RESULT_SUFFIXES]
        if not files:
            print("Nothing to pack yet: no figure has been drawn in this session. "
                  "Run the plot cells above first.")
            return

        archive = EXPORT_DIR / f"{NOTEBOOK_SLUG}_results.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f, f.relative_to(OUTPUT_DIR))

        total = sum(f.stat().st_size for f in files)
        print(f"{len(files)} file(s), {_human(total)} -> {_human(archive.stat().st_size)} compressed")
        for f in files:
            print(f"   {f.relative_to(OUTPUT_DIR)}")

        # A plain link would make JupyterLab try to *open* the zip, which fails with "is not
        # UTF-8 encoded". The /files/ endpoint hands it to the browser instead, and under
        # JupyterHub it has to carry the /user/<name>/ prefix or it reaches the hub and
        # returns an HTML error page.
        try:
            rel = archive.resolve().relative_to(Path.home())
        except ValueError:
            rel = archive.name
        _prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
        display(HTML(
            f'<p style="margin-top:8px"><a href="{_prefix}/files/{rel}?download=1" download '
            f'target="_blank" style="padding:8px 14px;background:#1B4053;color:white;'
            f'text-decoration:none;border-radius:5px;display:inline-block">'
            f'Download {archive.name}</a></p>'
            f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
            f'<code>{archive.name}</code> in the file browser on the left and choose '
            f'<i>Download</i>.</p>'))


bundle_btn.on_click(_make_bundle)
display(widgets.VBox([bundle_btn, bundle_out]))
print("Press 'Create archive' when you want to keep the figures of this run.")